# EMSCA Functions

**Author:** Dev Mehta and Mohammad Nour

**Description:** This notebook contains functions and utilities for EMSCA.

In [ ]:
import pytrinamic
from pytrinamic.connections import ConnectionManager
from pytrinamic.modules import TMCM6110
import time
import scipy.io
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import trange
from itertools import chain
from matplotlib.ticker import FormatStrFormatter

# Motor Control 

In [ ]:
class XYZ():
    def __init__(self):
        self.connectionManager = ConnectionManager()
        self.interface = self.connectionManager.connect()

        # Create an instance of the TMCM_6110 class
        self.module = TMCM6110(self.interface)


        self.motor_0 =  self.module.motors[0]
        self.motor_1 =  self.module.motors[1]
        self.motor_2 =  self.module.motors[2]
        print("Preparing parameters")
        
    def XYZ_setup(self,max_current=500,standby_current=200,boost_current=0,velocity=1500,acceleration=1000,position=0):
        motors_list = []
        for i in [0,1,2]:
            motor_name = f"motor_{i}"
            motor = getattr(self, motor_name)
            # Now you can use 'motor' as a reference to self.motor_0, self.motor_1, etc.
            motor.drive_settings.max_current=max_current
            motor.drive_settings.standby_current=standby_current
            motor.drive_settings.boost_current=boost_current
            motor.drive_settings.microstep_resolution = motor.ENUM.microstep_resolution_256_microsteps
            motor.max_acceleration=acceleration
            motor.max_velocity=velocity
#             motor.actual_position=position
            print(motor)
            motors_list.append(motor)
        return motors_list[2],motors_list[1],motors_list[0],self.interface
        
  

# Capturing functions


In [ ]:
def capture_location(scope, pt_exp, exp_name, x, y, num=1000):
    """
    Captures and stores traces for a given (x, y) location.

    Parameters:
    scope: Object responsible for capturing traces.
    pt_exp: Experiment dataset handler for plaintexts and keys.
    exp_name: Dataset handler where captured traces will be stored.
    x, y: Coordinates representing the capture location.
    num (int, optional): Number of traces to capture (default is 1000).
    """

    # Retrieve key, random plaintext, and fixed plaintext datasets
    keys_pt = pt_exp.get_dataset("keys").read_data(0, num)
    random_pt = pt_exp.get_dataset("plaintexts").read_data(0, num)
    fixed_pt = pt_exp.get_dataset("fixed_pt").read_data(0, num)

    # Capture traces using test vector leakage assessment (TVLA) method
    f, r = scope.capture_traces_tvla(num, keys_pt, fixed_pt, keys_pt, random_pt)

    # Store captured traces for the given location
    print("Storing for location: " + str(x) + "_" + str(y))
    exp_name.add_dataset("fixed_" + str(x) + "_" + str(y), f, datatype="float32")
    exp_name.add_dataset("random_" + str(x) + "_" + str(y), r, datatype="float32")

    print("Traces stored")

    return None


def Grid_Tracing_scapegoat(X_range, Y_range, X_number_of_step, Y_number_of_step, X, Y, Z, interface, scope, pt_exp, exp_store, number_of_traces):
    """
    Performs grid-based scanning and captures traces at each step.

    Parameters:
    X_range, Y_range: Step sizes for movement in X and Y directions.
    X_number_of_step, Y_number_of_step: Number of steps to take in X and Y directions.
    X, Y, Z: Actuators controlling movement along respective axes.
    interface: Communication interface for device control.
    scope: Object responsible for capturing traces.
    pt_exp: Experiment dataset handler for plaintexts and keys.
    exp_store: Experiment object to store captured traces.
    number_of_traces: Number of traces to capture at each grid point.
    """

    cordinate_traces = {}  # Dictionary to store traces at different coordinates
    X_moment = 0
    Y_moment = 0

    # Capture initial location traces
    capture_location(scope, pt_exp, exp_store, X_moment, Y_moment, number_of_traces)

    # Store initial positions
    X_start_position = X.get_actual_position()
    Y_start_position = Y.get_actual_position()
    print(f"Starting Position - ({X_start_position}, {Y_start_position})")

    # Perform scanning along the Y-axis
    while Y_moment <= Y_number_of_step:
        X_initial_position = X.get_actual_position()
        Y_initial_position = Y.get_actual_position()

        # Move in the positive X direction
        for _ in range(X_number_of_step):
            X.move_by(X_range)
            print(f'Moving X to {X_initial_position + X_range}')
            while X.get_actual_position() != X_initial_position + X_range:
                time.sleep(0.1)  # Wait until movement is complete
            X_moment += 1
            capture_location(scope, pt_exp, exp_store, X_moment, Y_moment, number_of_traces)
            X_initial_position = X.get_actual_position()

        if Y_moment == Y_number_of_step:
            break  # Stop if the last Y step is reached

        # Move in the positive Y direction
        Y.move_by(Y_range)
        Y_moment += 1
        print(f'Moving Y to {Y_initial_position + Y_range}')
        while Y.get_actual_position() != Y_initial_position + Y_range:
            time.sleep(0.1)  # Wait until movement is complete
        capture_location(scope, pt_exp, exp_store, X_moment, Y_moment, number_of_traces)
        Y_initial_position = Y.get_actual_position()

        # Move in the negative X direction
        for _ in range(X_number_of_step):
            X.move_by(-X_range)
            print(f'Moving X to {X_initial_position - X_range}')
            while X.get_actual_position() != X_initial_position - X_range:
                time.sleep(0.1)  # Wait until movement is complete
            X_moment -= 1
            capture_location(scope, pt_exp, exp_store, X_moment, Y_moment, number_of_traces)
            X_initial_position = X.get_actual_position()

        if Y_moment == Y_number_of_step:
            break  # Stop if the last Y step is reached

        # Move in the positive Y direction again
        Y.move_by(Y_range)
        Y_moment += 1
        print(f'Moving Y to {Y_initial_position + Y_range}')
        while Y.get_actual_position() != Y_initial_position + Y_range:
            time.sleep(0.1)  # Wait until movement is complete
        capture_location(scope, pt_exp, exp_store, X_moment, Y_moment, number_of_traces)
        Y_initial_position = Y.get_actual_position()

    # Return to the starting position
    X.move_to(X_start_position)
    while X.get_actual_position() != X_start_position:
        time.sleep(0.1)  # Wait until movement is complete

    Y.move_to(Y_start_position)
    while Y.get_actual_position() != Y_start_position:
        time.sleep(0.1)  # Wait until movement is complete

    print(f"Final Position - ({X.get_actual_position()}, {Y.get_actual_position()})")

    return None


# Metrics (one-click)

In [ ]:
def plot_CEMA_heatmap(test, pt_exp, num, target_byte=0, grid_size=5,rotation_n = 0,lower_b = 0,upper_b = 10000, model = 1,fntsz=18):
    """
    Compute and visualize Correlation Electromagnetic Analysis (CEMA) results as a heatmap.

    Parameters:
    - test: An object that provides access to trace datasets.
    - pt_exp: Experiment dataset handler for keys and plaintexts.
    - num: Number of traces to process.
    - target_byte: Byte index for correlation analysis (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    
    Returns:
    - CEMA_guesses_rotated: Rotated array of best key guesses from CEMA.
    - CEMA_values_rotated: Rotated array of maximum correlation values from CEMA.
    """

    # Retrieve keys and plaintext datasets
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

    # Initialize arrays to store CEMA results
    CEMA_values = np.zeros((grid_size, grid_size))  # Stores maximum correlation values
    CEMA_guesses = np.zeros((grid_size, grid_size))  # Stores corresponding key guesses

    # Perform CEMA analysis across the grid
    for i in range(grid_size):
        for j in trange(grid_size):
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)
            upper_b = len(traces[0])
            # Perform CEMA to obtain correlation values and best key guess
            
            best_guess, max_correlation = scapegoat_cpa_byte(traces[:,lower_b:upper_b], keys, plaintexts, target_byte)
            
                
            # Store results
            CEMA_values[i, j] = max_correlation
            CEMA_guesses[i, j] = best_guess

    # Rotate the heatmap for correct visualization
    CEMA_values_rotated = np.rot90(CEMA_values, k=rotation_n )  # Rotate by 90 degrees clockwise
    CEMA_guesses_rotated = np.rot90(CEMA_guesses, k=rotation_n )

    # Create a heatmap visualization
    plt.figure(figsize=(8, 6))
    sns.heatmap(CEMA_values_rotated, annot=True, cbar=True, square=True)

    # Add labels and title
    plt.title("CEMA Heatmap",fontsize=fntsz+4)
    plt.xlabel("Grid Column (j)",fontsize=fntsz+2)
    plt.ylabel("Grid Row (i)",fontsize=fntsz+2)
    plt.yticks(fontsize=fntsz)
    plt.xticks(fontsize=fntsz)

    # Display the plot
    plt.show()

    return CEMA_guesses_rotated, CEMA_values_rotated
def plot_t_statistic_heatmap(test, grid_size=5,rotation_n = 0,fntsz=20,title = False):
    """
    Compute and visualize t-statistics as a heatmap.

    This function calculates t-statistics for each position in a grid 
    and generates a heatmap representation.

    Parameters:
    - test: An object that has a `calculate_t_test` method to compute the t-statistics.
    - grid_size: The size of the grid (default is 5x5).

    Returns:
    - t_values_rotated: Rotated array of maximum absolute t-statistics.
    """

    # Initialize the t-statistics array
    t_values = np.zeros((grid_size, grid_size))

    # Compute t-statistics for each grid position
    for i in range(grid_size):
        for j in range(grid_size):
            t_stat, t_max = test.calculate_t_test(f"fixed_{i}_{j}", f"random_{i}_{j}")
            t_values[i, j] = np.nanmax(np.abs(t_stat))  # Store the maximum absolute t-statistic

    # Rotate the heatmap for correct visualization
    t_values_rotated = np.rot90(t_values, k=rotation_n)  # Rotate by 90 degrees clockwise

    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(t_values_rotated, annot=True, cbar=True, square=True)

    # Add labels and title
    if title:
        plt.title("Heatmap of t-statistics",fontsize=fntsz+4)
    plt.xlabel("Grid Column (j)",fontsize=fntsz+2)
    plt.ylabel("Grid Row (i)",fontsize=fntsz+2)
    plt.yticks(fontsize=fntsz)
    plt.xticks(fontsize=fntsz)
    
    
    # Display the plot
#     plt.savefig(f"final_data//{config}_t_heatmap.png",format="png",bbox_inches="tight")
    fig= plt.gcf()
    fig.savefig("results/t_statistic_heatmap.png", bbox_inches="tight", dpi=300)
    plt.show()

    return t_values_rotated,fig

def reverse_coords_ccw(x_rot, y_rot, shape, k):
    """
    Given a point (x_rot, y_rot) in an array that was created as
        rotated = np.rot90(original, k)
    (i.e., original was rotated counter-clockwise k times),
    return the corresponding (x_orig, y_orig) in the original array.

    Parameters
    ----------
    x_rot, y_rot : int
        Coordinates in the rotated image.
    shape : tuple
        Shape of the original array as (rows, cols).
    k : int
        The `k` passed to np.rot90 (number of 90° CCW rotations applied).

    Returns
    -------
    (x_orig, y_orig)
    """
    # np.rot90 with k CCW is equivalent to (4 - k) clockwise rotations.
    n_clockwise = (k) % 4  # same as (4 - k) % 4
    rows, cols = shape
    if n_clockwise == 0:
        return x_rot, y_rot
    elif n_clockwise == 1:  # 90° clockwise
        return y_rot, cols - 1 - x_rot
    elif n_clockwise == 2:  # 180°
        return rows - 1 - x_rot, cols - 1 - y_rot
    elif n_clockwise == 3:  # 90° counter-clockwise
        return rows - 1 - y_rot, x_rot


In [ ]:
def MI_heatmap(test,labels , num ,nyst, landmarks , grid_size=5,rotation_n = 2,lower_b = 0,upper_b = 10000):
    HX = np.zeros((grid_size,grid_size))

    for i in range(grid_size):
        for j in range(grid_size):
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)
            traces = traces[: , lower_b:upper_b]
            
            
            
            HX[i, j] = HXX(traces ,labels , 1.01 , nyst , landmarks)
            
            
    HX_rotated = np.rot90(HX, k=rotation_n )       
    MI = wienner(HX_rotated)

    

    plt.figure(figsize=(18, 16))
    # sns.heatmap(MI, annot=True, fmt=".2f", cbar=True, square=True)
    ax = sns.heatmap(MI, annot=True, fmt=".2f", cbar=True, square=True)

    # --- Access the colorbar ---
    cbar = ax.collections[0].colorbar

    ax.collections[0].set_clim(0.0, 1)

    # --- 2. Set ticks manually ---
    ticks = np.linspace(0.0, 1, 5)   # 7 ticks from 0 to 3
    cbar.set_ticks(ticks)

    # --- 3. Format tick labels ---
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

    plt.xlabel("Grid Column (j)")
    plt.ylabel("Grid Row (i)")

    fig= plt.gcf()
    fig.savefig("results/mi_heatmap.png", bbox_inches="tight", dpi=300)
    plt.show()

    return MI , HX_rotated


## helper functions  

In [ ]:
import numpy as np
from scipy.stats import norm


def scapegoat_cpa_byte(traces, keys, plaintexts, target_byte):
    """
    Perform Correlation Power Analysis (CPA) on a specific key byte to guess the subkey.

    Parameters:
    - traces: The captured power traces.
    - keys: The actual secret keys corresponding to the traces.
    - plaintexts: The plaintexts used for the power analysis.
    - target_byte: The index of the target byte in the key to analyze.

    Returns:
    - best_guess: The best guess for the target key byte.
    - cpa_ref: The highest correlation value obtained for the target byte.
    """
    max_cpa = np.zeros(256)  # Store maximum CPA values for each possible subkey guess
    cpa_ref = 0  # Store highest correlation value for the target byte
    best_guess = 0  # Store best subkey guess for the target byte

    # Perform CPA attack for each possible subkey guess (0-255)
    for k in range(256):
        # Compute leakage model for each subkey guess
        leakage = leakage_model_hamming_weight(
            num_traces=len(plaintexts),
            plaintexts=plaintexts,
            subkey_guess=k,
            target_byte=target_byte
        )
        # Compute the Pearson correlation between the leakage and the traces
        correlation = pearson_correlation(leakage, traces)
        max_cpa[k] = np.nanmax(np.abs(correlation))  # Store the highest correlation for this guess

    # Find the best subkey guess and highest correlation value
    best_guess = np.argmax(max_cpa)
    cpa_ref = np.nanmax(max_cpa)

    return best_guess, cpa_ref

def test_to_avg(test,test_avg,avg=10,grid_size =11):

    for i in range(grid_size):
        for j in trange(grid_size):
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            rand = test.get_dataset(f"random_{i}_{j}").read_all()
            fix = test.get_dataset(f"fixed_{i}_{j}").read_all()
            
            # Reshape so each group of 10 rows becomes one block
            f = fix.reshape(-1, avg, fix.shape[1]).mean(axis=1)
            r = rand.reshape(-1, avg, rand.shape[1]).mean(axis=1)
            
            test_avg.add_dataset("fixed_" + str(i) + "_" + str(j), f, datatype="float32")
            test_avg.add_dataset("random_" + str(i) + "_" + str(j), r, datatype="float32")
            
def test_to_downsample(test, test_avg, step=10, grid_size=11):

    for i in range(grid_size):
        for j in trange(grid_size):
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            rand = test.get_dataset(f"random_{i}_{j}").read_all()
            fix = test.get_dataset(f"fixed_{i}_{j}").read_all()
            
            # Select every 10th row instead of averaging
            f = fix[::step]
            r = rand[::step]
            
            test_avg.add_dataset(f"fixed_{i}_{j}", f, datatype="float32")
            test_avg.add_dataset(f"random_{i}_{j}", r, datatype="float32")

def plot_CEMA_traces_temp(traces, pt_exp,num, target_byte=0, div=10, visualize_correct = True,model =1, r1 = 9, r2 = 8,col_tar = 'sbb_o6',title= False):
    """
    Generate a CPA correlation plot comparing the correct key vs. wrong keys over increasing trace counts.

    This function performs Correlation Power Analysis (CPA) across different numbers of traces, 
    visualizing how the correct key and wrong keys' correlation evolve.

    Parameters:
    - test: An object that provides trace datasets.
    - pt_exp: Experiment data handler providing plaintext and key datasets.
    - num: Total number of traces to analyze.
    - target_byte: The target byte index in the key (default is 0).
    - x, y: Grid position for selecting the dataset.
    - div: Step size for processing traces in intervals.

    Returns:
    - maxcpa_matrix: A matrix storing the maximum CPA correlation values for all 256 key guesses.
    """

#     # Load key and plaintext data
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

#     # Load power traces for the selected grid position
#     traces = test.get_dataset(f"random_{x}_{y}").read_data(0, num)
    correct_key = keys[0][target_byte]
    # Initialize a matrix to store max CPA values across different key guesses
    maxcpa_matrix = np.zeros((int(num / div), 256))


    iterations = 1
    for i in trange(1, num):
        if i % div == 0:
            if visualize_correct:
                k = correct_key
                if model ==1:
                    leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)
                elif model ==2:
                    c = AES(keys[0])

                    leakage = hd_cpa_last_round(plaintext=plaintexts,num_traces=i,c=c,target_byte=target_byte,r1=r1,r2=r2)
              
                # Compute Pearson correlation
                correlation = pearson_correlation(leakage, traces[:i])

                # Store max correlation value for this key guess
                maxcpa_matrix[int(i / div), k] = np.nanmax(np.abs(correlation))
            else:
                for k in range(256):
                    # Compute leakage model using Hamming weight
                    leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)

                    # Compute Pearson correlation
                    correlation = pearson_correlation(leakage, traces[:i])

                    # Store max correlation value for this key guess
                    maxcpa_matrix[int(i / div), k] = np.nanmax(np.abs(correlation))

                iterations += 1

    # Debugging: Check matrix shape
    print(target_byte)
    print("Shape of maxcpa_matrix:", maxcpa_matrix.shape)

    if visualize_correct:
        xp =  np.arange(2,  maxcpa_matrix.shape[0])
    else:
        xp = np.arange(2, min(iterations + 2, maxcpa_matrix.shape[0]))

    # Plotting
    plt.figure(figsize=(10, 6))

    if maxcpa_matrix.shape[0] > 2:
        # Plot the statistical threshold
        plt.plot(xp, (abs(4) / np.sqrt(xp * div)) * np.ones_like(xp), color="black", linestyle='dotted', linewidth=1.5, label="Threshold")

        # Plot CPA correlations for all 256 key hypotheses
        if visualize_correct:
            plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, int(correct_key)], color="red",    alpha=0.9, linewidth=1.5, label="Correct key")
        else:
            for i in range(256):
                if i == correct_key:  # Assuming 43 is the correct key
                    plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="red",
                             alpha=0.9, linewidth=1.5, label="Correct key")
                else:
                    plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="grey",
                             alpha=0.1, linewidth=0.5, label="Wrong keys" if i == 0 else "")

    # Configure plot labels and title
    plt.xlabel(f"No. of traces × {div}", fontsize=20)
    plt.ylabel("Max CPA Value", fontsize=20)
    if title:
        plt.title(f"Correlation Power Analysis", fontsize=24)
    plt.yticks(fontsize=18)
    plt.xticks(fontsize=18)
    plt.legend()
    fig = plt.gcf()
    # Display the plot
    plt.show()

    if visualize_correct:
        plt.figure(figsize=(10, 6))
        plt.plot(correlation)
        plt.xlabel(f"Time Samples", fontsize=20)
        plt.ylabel("CPA Value", fontsize=20)
        if title:
            plt.title(f"Correlation Power Analysis - (byte {target_byte})", fontsize=24)
        plt.yticks(fontsize=18)
        plt.xticks(fontsize=18)
        plt.ylim(-0.3, 0.3) 
        plt.legend()

        # Display the plot
        plt.show()
    # --- Compute threshold crossing index ---
    cpa_curve = maxcpa_matrix[2:len(xp) + 2, int(correct_key)]
    threshold_curve = (abs(4) / np.sqrt(xp * div))

    # Find the first index where CPA exceeds threshold AND stays above afterwards
    cross_idx = None
    for idx in range(len(cpa_curve)):
        if cpa_curve[idx] >= threshold_curve[idx]:
            # Check if it stays above for all later points
            if np.all(cpa_curve[idx:] >= threshold_curve[idx:]):
                cross_idx = idx
                break

    if cross_idx is not None:
        min_traces_to_cross = xp[cross_idx] * div
        print(f"✅ CPA crosses threshold permanently at index {cross_idx} (≈ {min_traces_to_cross} traces)")
    else:
        min_traces_to_cross = None
        print("⚠️ CPA never stays permanently above the threshold.")

    return maxcpa_matrix, correlation,fig,min_traces_to_cross



In [ ]:
import numpy as np
from scipy.stats import norm
def hd_cpa_last_round(plaintext,num_traces,c,target_byte=0,r1=9,r2=8):
#     c = AES(keys)
    
    leakage = np.empty(num_traces, dtype=object)
#     print(r1,r2)
    for i in range(num_traces):
        a,b,r_1,r_2 = c.encrypt(plaintext[i],round_1=r1,round_2=r2)
        if r1==10:
#             print("reached round 10")
            leakage[i] = bin(a[target_byte] ^ b[target_byte]).count('1')
        elif r1!=10:
#             print("i dont listen")
            leakage[i] = bin(r_1[target_byte] ^ r_2[target_byte]).count('1')
        
    return leakage


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.kernel_approximation import Nystroem  # outside the function, once

import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
from scipy.stats import pearsonr


import numpy as np
from scipy.spatial.distance import cdist
from numpy.linalg import eigh
import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat
from scipy.signal import wiener





def normalize_to_0_1(vals):
    vals = np.array(vals, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    return (vals - vmin) / (vmax - vmin)


def wienner(HXcell):



    # === Corners ===
    corners = np.array([
        HXcell[0, 0],
        HXcell[0, -1],
        HXcell[-1, 0],
        HXcell[-1, -1]
    ])

    # === Noise selection (same logic as MATLAB prctile) ===
    threshold = np.percentile(corners, 95)
    noise_samples = corners[corners < threshold]

    # === Noise stats ===
    mu_N = np.mean(noise_samples)
    sigma_N = np.std(noise_samples)

    # === Normalize ===
    HX_z_norm = HXcell - mu_N

    MI_masked = HX_z_norm.copy()
    MI_masked[HX_z_norm < 0] = 0

    # === Signal stats ===
    mu_L = np.mean(MI_masked)
    sigma_L = np.std(MI_masked)

    # === SNR estimate ===
    eps = 1e-12
    SNR_est = sigma_L / (sigma_N + eps)

    # === Window calculation ===
    win = (9 - 2 * SNR_est)

    # === Convert to grayscale (if needed) ===

    I = MI_masked

    # === Wiener filter (equivalent to wiener2) ===
    MI_masked_smooth = wiener(I, (3, 3))
    MI = normalize_to_0_1(MI_masked_smooth)
    
    return MI


def median_heuristic(X):
    """
    Compute sigma using the median heuristic.
    
    Parameters
    ----------
    X : ndarray of shape (n, d)
        Input data, n samples in d dimensions.
    
    Returns
    -------
    sigma : float
        Median heuristic bandwidth parameter.
    """
    X = np.asarray(X)
    n = X.shape[0]

    # Compute pairwise squared distances
    diff = X[:, None, :] - X[None, :, :]
    D2 = np.sum(diff**2, axis=2)

    # Take only the upper triangle (i<j) to avoid duplicates/zeros
    iu = np.triu_indices(n, k=1)
    sigma = np.sqrt(np.median(D2[iu]))
    return sigma

def effective_rank(K):
    """Effective rank = (sum λ)^2 / sum λ^2, using symmetric eigendecomp."""
    # eigh for symmetric matrices; ensure numerical non-negativity
    w = eigh(0.5*(K+K.T), UPLO='L')[0]
    w = np.clip(w, 0.0, None)
    s1 = np.sum(w)
    s2 = np.sum(w**2)
    return (s1**2 / s2) if s2 > 0 else 0.0

def moving_average(x, k=3):
    """Simple centered moving average (odd k). Falls back to same length."""
    if k <= 1 or k % 2 == 0:
        return x
    pad = k // 2
    xp = np.pad(x, (pad, pad), mode='edge')
    kernel = np.ones(k) / k
    return np.convolve(xp, kernel, mode='valid')

def select_landmarks_leverage(X_np, m, gamma, pilot_mult=4, ridge=1e-8, random_state=0):
    """
    X_np: (N, d) numpy array
    m:    number of landmarks to choose
    gamma: RBF gamma used for Nyström
    pilot_mult: pilot components ≈ 3–8× m works well
    ridge: small Tikhonov for stability
    """
    r = max(m * pilot_mult, m+1)
    pilot = Nystroem(kernel='rbf', gamma=gamma, n_components=r, random_state=random_state)
    Phi = pilot.fit_transform(X_np)                           # (N, r)

    # leverage_i ≈ phi_i^T (Phi^T Phi + λI)^{-1} phi_i
    G = Phi.T @ Phi
    M = np.linalg.inv(G + ridge * np.eye(G.shape[0]))        # (r, r)
    A = Phi @ M                                              # (N, r)
    leverage = np.sum(A * Phi, axis=1)                       # diag(Phi M Phi^T)

    # pick top-m (or sample ∝ leverage)
    idx = np.argsort(leverage)[-m:]
    return np.sort(idx), leverage

def cel_Hx(i,j, sig):
    traces = np.load(f'/home/mohammadn/IT_in_SCA/dataset_Dev/impl_3/random_{10-i}_{10-j}.npy')
    mi=[]
    for j in range(traces.shape[1]):
        X =100*traces[:100,j]
  

        mi.append(np.array(HXX(X[:100],sig,1.01 )))
        
    return np.array(mi)

def calculate_gram_mat(x, sigma):
    """calculate gram matrix for variables x
        Args:
        x: random variable with two dimensional (N,d).
        sigma: kernel size of x (Gaussain kernel)
    Returns:
        Gram matrix (N,N)
    """
    x = x.view(x.shape[0],-1)
    instances_norm = torch.sum(x**2,-1).reshape((-1,1))
    dist= -2*torch.mm(x,x.t()) + instances_norm + instances_norm.t()
    return torch.exp(-dist /sigma)


def renyi_entropy(x,sigma,alpha):
    
    """calculate entropy for single variables x (Eq.(9) in paper)
        Args:
        x: random variable with two dimensional (N,d).
        sigma: kernel size of x (Gaussain kernel)
        alpha:  alpha value of renyi entropy
    Returns:
        renyi alpha entropy of x. 
    """
    
    k = calculate_gram_mat(x,sigma)
    
    
    
    
    eps = 1e-5  # or tune this
    k += eps * torch.eye(k.shape[0], device=k.device)

    


    k = k/torch.trace(k) 
    eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part

    eigv = torch.abs(eigv)  
    eig_pow = eigv**alpha
    entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
    return entropy,k

def renyi_entropy_nystrom(x,sigma,alpha,landmarks):
    
    """calculate entropy for single variables x (Eq.(9) in paper)
        Args:
        x: random variable with two dimensional (N,d).
        sigma: kernel size of x (Gaussain kernel)
        alpha:  alpha value of renyi entropy
    Returns:
        renyi alpha entropy of x. 
    """
    
    # k = calculate_gram_mat(x,sigma)
    
    ## nystrom with leverage score


    # X_np = x.detach().cpu().numpy()
    # U, _, _ = np.linalg.svd(X_np[:700], full_matrices=False)  
    # lev_scores = np.sum(U**2, axis=1)
    # landmark_idx = np.random.choice(len(lev_scores), size=landmarks, p=lev_scores/lev_scores.sum(), replace=False)
    
    # landmarks_ = X_np[landmark_idx]
    # feature_map = Nystroem(kernel='rbf', gamma=1.0/sigma, n_components=landmarks)
    # feature_map.fit(landmarks_)  
    # Phi_np = feature_map.transform(X_np[:800])
    # k = torch.from_numpy(Phi_np).to(x.device)
    # k = k @ k.T


    ##
    
    
    # nystrom 
    
    feature_map = Nystroem(kernel='rbf', gamma=1.0/float(sigma), n_components=landmarks)

    # line 2: get Φ (N × m) from Nyström
    Phi_np = feature_map.fit_transform(x.detach().cpu().numpy())

    # line 3: approximate Gram matrix K ≈ Φ Φᵀ in torch
    k = torch.from_numpy(Phi_np).to(x.device)
    k = k @ k.T
    
    #  nystrom 
    
    
    
    
    
    eps = 1e-5  # or tune this
    k += eps * torch.eye(k.shape[0], device=k.device)

    


    k = k/torch.trace(k) 
    eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part

    eigv = torch.abs(eigv)  
    eig_pow = eigv**alpha
    entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
    return entropy


def k_cal(x):
    m = x.shape[0]
    classes = torch.unique(x)
    C = len(classes)
    L = torch.zeros((m, C), dtype=torch.float32)
    for c, class_label in enumerate(classes):
        idx = (x == class_label)
        n_c = idx.sum()
        if n_c > 0:
            L[idx, c] = 1.0 / torch.sqrt(n_c.float())
    # Compute Kx = L * L^T
    k = L @ L.T  # Matrix multiplication
    return k



def renyi_entropy_labels(x,sigma,alpha):
    """calculate entropy for single variables x (Eq.(9) in paper)
        Args:
        x: random variable with two dimensional (N,d).
        sigma: kernel size of x (Gaussain kernel)
        alpha:  alpha value of renyi entropy
    Returns:
        renyi alpha entropy of x.
    """
    m = x.shape[0]
    classes = torch.unique(x)
    C = len(classes)
    L = torch.zeros((m, C), dtype=torch.float32)
    for c, class_label in enumerate(classes):
        idx = (x == class_label)
        n_c = idx.sum()
        if n_c > 0:
            L[idx, c] = 1.0 / torch.sqrt(n_c.float())
    # Compute Kx = L * L^T
    k = L @ L.T  # Matrix multiplication
    # Normalize trace to 1
    k = k / torch.trace(k)
    eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part
    eigv = torch.abs(eigv)
    eig_pow = eigv**alpha
    entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
    return entropy



def joint_entropy(x,y,s_x,s_y,alpha):
    """calculate joint entropy for random variable x and y (Eq.(10) in paper)
        Args:
        x: random variable with two dimensional (N,d).
        y: random variable with two dimensional (N,d).
        s_x: kernel size of x
        s_y: kernel size of y
        alpha:  alpha value of renyi entropy
    Returns:
        joint entropy of x and y.
    """
    x = calculate_gram_mat(x,s_x)
    # y = calculate_gram_mat(y,s_y)
    y = k_cal(y)
    k = torch.mul(x,y)
    # eps = 1e-5  # or tune this
    # k += eps * torch.eye(k.shape[0], device=k.device)
    k = k/torch.trace(k)
    eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part
    eigv = torch.abs(eigv)
    eig_pow =  eigv**alpha
    entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
    return entropy



def calculate_MI(x,y,alpha,s_x,s_y,normalize):
    """calculate Mutual information between random variables x and y
    Args:
        x: random variable with two dimensional (N,d).
        y: random variable with two dimensional (N,d).
        s_x: kernel size of x
        s_y: kernel size of y
        normalize: bool True or False, noramlize value between (0,1)
    Returns:
        Mutual information between x and y (scale)
    """
    Hx , kx = renyi_entropy(x,sigma=s_x , alpha=alpha)
    # Hy = renyi_entropy(y,sigma=s_y , alpha=alpha)
    Hy = renyi_entropy_labels(y,sigma=s_y , alpha=alpha)

    Hxy= joint_entropy(x,y,s_x,s_y , alpha=alpha)
    if normalize:
        Ixy = Hx+Hy-Hxy
        Ixy = Ixy/(torch.max(Hx,Hy))
    else:
        Ixy = Hx+Hy-Hxy
    return Ixy,Hx, Hxy, kx

def get_sigma(dim, n, std):
    h = (0.9*std)/(n**1.5)
    return h*n**(-1/(4+dim))


def MBRE(X , Y, sig):
    
    mis = []
    for i in range(X.shape[1]):
        
        x = X[:100,i]


        
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(np.array(Y, dtype=int), dtype=torch.float32)



        # sigma_x = get_sigma(1,100,torch.std(x) )
        # sigma_y = get_sigma(1,000,torch.std(y) )
        # mio = calculate_MI(x , y,1.01, 0.6,1,False)     # alpha was 0.6 for the first dataset impl_1
        mis.append( calculate_MI(100*x , y[:100],1.01,sig,1,False))
    
    return np.array(mis)



def HXX(X ,labels , alpha , nyst ,  landmarks):
    variances = np.var(X, axis=0)
    
    mask = variances >0
    
    traces = X[:, mask]
    
    X = traces
    m = X.shape[0]
    
    # 1) pairwise squared distances
    D2 = cdist(X, X, metric='euclidean')**2
    
    upper = D2[np.triu_indices(m, k=1)]
    
    sigma0 = np.sqrt(0.5*np.median(upper)) if upper.size > 0 else 1.0
    
    span_low = 1/3
    span_high = 10
    n_sigma = 20
    
    sigmas = np.logspace(np.log10(sigma0*span_low),
                         np.log10(sigma0*span_high),
                         num=n_sigma)
    
    eff_ranks = []
    MI_vals = []
    valid_sigmas =[]
    Hx_ = []
    Hxy_ =[]
    
    
    for i in range(n_sigma):


        sigma_x = sigmas[i]


        h = np.array(labels, dtype=int)

        x = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(h, dtype=torch.float32)




        mio,Hx, Hxy, kx = calculate_MI(x , y,1.01, sigma_x,1,False)   
        Hy = Hxy + mio -Hx
#         if (Hxy > max(Hx,Hy) and mio < min(Hx,Hy) and mio>0 ):
        if (Hxy >= max(Hx,Hy) and mio>=0 ):
            eff_ranks.append(effective_rank(kx))
            MI_vals.append(mio)
            valid_sigmas.append(sigma_x)
            Hx_.append(Hx)
            Hxy_.append(Hxy)
            
            
    eff_ranks = np.array(eff_ranks)
    MI_vals = np.array(MI_vals)
    sigmas = np.array(valid_sigmas)
    smooth_k = 5
    
    rank_smooth = moving_average(eff_ranks, k=smooth_k)
    

    MI_smooth   = moving_average(MI_vals,   k=smooth_k)
    
    
    eps = 1e-12
    d1 = np.gradient(np.log(rank_smooth + eps))
    d2 = np.gradient(d1)
    elbow_idx = int(np.argmin(d2))
    mi_peak_idx = int(np.argmax(MI_smooth))
    sigma_opt_idx = min(elbow_idx, mi_peak_idx)
    sigma_opt = float(sigmas[sigma_opt_idx])
    
    
    if nyst:
        Hxfinal = renyi_entropy_nystrom(x,sigma_opt,1.01, landmarks)
        
    else:
        Hxfinal,_ =renyi_entropy(x,sigma_opt,1.01)
        
        

    
    return Hxfinal
    